In [10]:
print('hallo')

hallo


In [11]:
# uv pip install packaging psutil ninja
# MAX_JOBS=4 uv pip install -r requirements.txt --no-build-isolation

In [12]:
from esm.models.esmc import ESMC
from esm.sdk.api import ESMProtein, LogitsConfig

protein = ESMProtein(sequence="AAAAASEES")
client = ESMC.from_pretrained("esmc_300m").to("cuda") # or "cpu"
protein_tensor = client.encode(protein)


In [4]:
logits_output = client.logits(
   protein_tensor, LogitsConfig(sequence=True, return_embeddings=True)
)
print(logits_output.logits, logits_output.embeddings)

ForwardTrackData(sequence=tensor([[[-36.7500, -36.7500, -36.7500,  11.5625,  18.6250,  18.7500,  19.0000,
           18.7500,  18.5000,  19.0000,  18.3750,  18.3750,  18.1250,  18.6250,
           18.6250,  18.2500,  17.8750,  17.6250,  17.8750,  17.7500,  18.3750,
           16.5000,  17.1250,  16.2500,  17.7500,   2.0000,  -1.7109,   0.9297,
          -18.7500, -36.7500, -36.7500, -36.7500, -36.7500, -37.0000, -36.7500,
          -36.7500, -36.7500, -36.7500, -37.0000, -36.7500, -36.7500, -37.0000,
          -36.7500, -36.7500, -36.7500, -36.7500, -36.7500, -36.7500, -37.0000,
          -36.7500, -36.7500, -36.7500, -36.7500, -37.0000, -37.0000, -36.7500,
          -36.7500, -36.7500, -36.7500, -36.7500, -37.0000, -36.7500, -36.7500,
          -36.7500],
         [-39.5000, -39.5000, -39.5000,   6.7812,  16.3750,  18.0000,  16.1250,
           17.0000,  16.7500,  16.3750,  15.8125,  16.2500,  15.3750,  15.4375,
           16.3750,  15.5000,  15.0000,  14.6250,  14.7500,  14.5000,  18

## Quantization

In [13]:
from torchao.quantization import Int8DynamicActivationInt8WeightConfig, Int8WeightOnlyConfig, quantize_
import copy
import time

import torch
from esm.utils.sampling import _BatchedESMProteinTensor
from tqdm.auto import tqdm

quantized_client = copy.deepcopy(client)
quantize_(quantized_client, Int8DynamicActivationInt8WeightConfig())

weight_only_client = copy.deepcopy(client)
quantize_(weight_only_client, Int8WeightOnlyConfig())


/venv/main/lib/python3.12/site-packages/torchao/quantization/quant_api.py:936: UserWarning: Config Deprecation: version 1 of Int8WeightOnlyConfig is deprecated and will no longer be supported in a future release, please use version 2, see https://github.com/pytorch/ao/issues/2752 for more details
  warnings.warn(


In [ ]:

benchmark_sequences = [
    "MLEPLPCWDAAKDLKEPQCPPGDRVGVQPGNSRVWQGTMEKAGLAWTRGTGVQSEGTWESQRQDSDALPSPELLPQDQDKPFLRKACSPSNIPAVIITDMGTQEDGALEETQGSPRGNLPLRKLSSSSASSTGFSSSYEDSEEDISSDPERTLDPNSAFLHTLDQQKPRVSKSWRKIKNMVHWSPFVMSFKKKYPWIQLAGHAGSFKAAANGRILKKHCESEQRCLDRLMVDVLRPFVPAYHGDVVKDGERYNQMDDLLADFDSPCVMDCKMGIRTYLEEELTKARRKPSLRKDMYQKMIEVDPEAPTEEEKAQRAVTKPRYMQWRETISSTATLGFRIEGIKKEDGTVNRDFKKTKTREQVTEAFREFTKGNHNILIAYRDRLKAIRTTLEVSPFFKCHEVIGSSLLFIHDKKEQAKVWMIDFGKTTPLPEGQTLQHDVPWQEGNREDGYLSGLNNLVDILTEMSQDAPLA",
    "MSLSTEQMLRDYPRSMQINGQIPKNAKELAEKLRNQDVNAAIATIKTQLAQFNEVVGQYDSDVGAGMVSFMSLSTEQMLRDYPRSMQINGQIPKNAKELAEKLRNQDVNAAIATIKTQLAQFNEVVGQYDSDVGAGMVSFMSLSTEQMLRDYPRSMQINGQIPKNAKELAEKLRNQDVNAAIATIKTQLAQFNEVVGQYDSDVGAGMVSFMSLSTEQMLRDYPRSMQINGQIPKNAKELAEKLRNQDVNAAIATIKTQLAQFNEVVGQYDSDVGAGMVSFMSLSTEQMLRDYPRSMQINGQIPKNAKELAEKLRNQDVNAAIATIKTQLAQFNEVVGQYDSDVGAGMVSFMSLSTEQMLRDYPRSMQINGQIPKNAKELAEKLRNQDVNAAIATIKTQLAQFNEVVGQYDSDVGAGMVSFMSLSTEQMLRDYPRSMQINGQIPKNAKELAEKLRNQDVNAAIATIKTQLAQFNEVVGQYDSDVGAGMVSF",
    "MNNRWILLLLLVAVAVTGMAQADVSGSEEKLQAKLEAAGVEVVKTLQDKVEELLSKNYHLENEVARLKKLVGERMNNRWILLLLLVAVAVTGMAQADVSGSEEKLQAKLEAAGVEVVKTLQDKVEELLSKNYHLENEVARLKKLVGERMNNRWILLLLLVAVAVTGMAQADVSGSEEKLQAKLEAAGVEVVKTLQDKVEELLSKNYHLENEVARLKKLVGERMNNRWILLLLLVAVAVTGMAQADVSGSEEKLQAKLEAAGVEVVKTLQDKVEELLSKNYHLENEVARLKKLVGERMNNRWILLLLLVAVAVTGMAQADVSGSEEKLQAKLEAAGVEVVKTLQDKVEELLSKNYHLENEVARLKKLVGERMNNRWILLLLLVAVAVTGMAQADVSGSEEKLQAKLEAAGVEVVKTLQDKVEELLSKNYHLENEVARLKKLVGER",
    "MTEITAAMVKELRESTGAGMMDCKNALSETQHEWAYKELGIKYDKDQQAANLAAQAEADKAAADMTEITAAMVKELRESTGAGMMDCKNALSETQHEWAYKELGIKYDKDQQAANLAAQAEADKAAADMTEITAAMVKELRESTGAGMMDCKNALSETQHEWAYKELGIKYDKDQQAANLAAQAEADKAAADMTEITAAMVKELRESTGAGMMDCKNALSETQHEWAYKELGIKYDKDQQAANLAAQAEADKAAADMTEITAAMVKELRESTGAGMMDCKNALSETQHEWAYKELGIKYDKDQQAANLAAQAEADKAAADMTEITAAMVKELRESTGAGMMDCKNALSETQHEWAYKELGIKYDKDQQAANLAAQAEADKAAADMTEITAAMVKELRESTGAGMMDCKNALSETQHEWAYKELGIKYDKDQQAANLAAQAEADKAAAD",
]

batched_protein_tensor = _BatchedESMProteinTensor(
    sequence=client._tokenize(benchmark_sequences)
)
logits_config = LogitsConfig(sequence=True, return_embeddings=True)


In [16]:
WARMUP_STEPS = 2
BENCHMARK_STEPS = 10
ENABLE_COMPILE = True
RUN_PROFILER = True
PROFILE_MODEL = "gpu_int8_weight_only"


def maybe_compile_model(model):
    if not ENABLE_COMPILE:
        return model
    return torch.compile(model, mode="max-autotune", fullgraph=True)


def benchmark_logits(model, inputs, label, num_steps=BENCHMARK_STEPS, warmup_steps=WARMUP_STEPS):
    model.eval()
    use_cuda = torch.device(inputs.device).type == "cuda"

    for _ in range(warmup_steps):
        _ = model.logits(inputs, logits_config)

    if use_cuda:
        torch.cuda.synchronize()

    start = time.perf_counter()
    for _ in tqdm(range(num_steps), desc=label):
        _ = model.logits(inputs, logits_config)

    if use_cuda:
        torch.cuda.synchronize()

    total_time = time.perf_counter() - start
    return {
        "label": label,
        "total_s": total_time,
        "avg_ms": 1000 * total_time / num_steps,
    }


gpu_results = []
gpu_models = {
    "gpu_fp": maybe_compile_model(client),
    "gpu_int8_dynamic": maybe_compile_model(quantized_client),
    "gpu_int8_weight_only": maybe_compile_model(weight_only_client),
}

print(f"Benchmark batch size: {batched_protein_tensor.sequence.shape[0]}")
print(f"Tokenized tensor shape: {tuple(batched_protein_tensor.sequence.shape)}")

for label, model in gpu_models.items():
    gpu_results.append(benchmark_logits(model, batched_protein_tensor, label))

baseline = gpu_results[0]["total_s"]
for result in gpu_results:
    result["speedup_vs_gpu_fp"] = baseline / result["total_s"]
    print(
        f"{result['label']}: total={result['total_s']:.2f}s | avg={result['avg_ms']:.2f} ms | speedup vs gpu_fp={result['speedup_vs_gpu_fp']:.2f}x"
    )

if RUN_PROFILER:
    profile_target = gpu_models[PROFILE_MODEL]
    with torch.profiler.profile(
        activities=[torch.profiler.ProfilerActivity.CPU, torch.profiler.ProfilerActivity.CUDA],
        record_shapes=True,
        profile_memory=True,
    ) as prof:
        _ = profile_target.logits(batched_protein_tensor, logits_config)
        torch.cuda.synchronize()
    print(f"Profiler target: {PROFILE_MODEL}")
    print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=20))


Benchmark batch size: 4
Tokenized tensor shape: (4, 474)


gpu_fp:   0%|          | 0/10 [00:00<?, ?it/s]

/venv/main/lib/python3.12/site-packages/torchao/dtypes/utils.py:89: UserWarning: Deprecation: PlainLayout is deprecated and will be removed in a future release of torchao, see https://github.com/pytorch/ao/issues/2752 for more details
  warnings.warn(
/venv/main/lib/python3.12/site-packages/torchao/dtypes/uintx/plain_layout.py:82: UserWarning: Deprecation: PlainAQTTensorImpl is deprecated and will be removed in a future release of torchao, see https://github.com/pytorch/ao/issues/2752 for more details
  warnings.warn(
/venv/main/lib/python3.12/site-packages/torchao/dtypes/affine_quantized_tensor.py:116: UserWarning: Deprecation: AffineQuantizedTensor is deprecated and will be removed in a future release of torchao, see https://github.com/pytorch/ao/issues/2752 for more details
  warnings.warn(


gpu_int8_dynamic:   0%|          | 0/10 [00:00<?, ?it/s]

gpu_int8_weight_only:   0%|          | 0/10 [00:00<?, ?it/s]

gpu_fp: total=1.01s | avg=101.18 ms | speedup vs gpu_fp=1.00x
gpu_int8_dynamic: total=2.62s | avg=261.66 ms | speedup vs gpu_fp=0.39x
gpu_int8_weight_only: total=1.19s | avg=118.99 ms | speedup vs gpu_fp=0.85x
Profiler target: gpu_int8_weight_only
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ---

## Pruning

In [ ]:
from torchao.sparsity.sparse_api import sparsify_, SemiSparseWeightConfig, semi_sparse_weight

import torch
import torch.nn as nn
from torchao.sparsity import sparsify_

def filter_fn(module: nn.Module, fqn: str) -> bool:
    return isinstance(module, nn.Linear)

# for 2:4 sparsity
m = sparsify_(client, semi_sparse_weight(), filter_fn)

/venv/main/lib/python3.12/site-packages/torch/sparse/semi_structured.py:571: UserWarning: The PyTorch API of SparseSemiStructuredTensor is in prototype stage and will change in the near future. Please open a Github issue for features requests and see our documentation on the torch.sparse module for further information about the project.
  return cls(


In [ ]:
client